# [수학 Retreat #6] 도형의 닮음: 무게중심의 역학과 무한 프랙탈

이 주피터 노트북은 중학교 2학년 과정의 **도형의 닮음** 단원을 파이썬 2D 그래픽 시각화를 통해 시각적으로 탐구하는 실습 교재입니다.
크기는 다를지라도 내면의 각도와 비율을 공유하는 **'본질적 동질성(Similarity)'**과 기하학적 팽팽함의 조화인 **'무게중심'**을 다룹니다.

### 🟢 본 실습에서 다룰 두 가지 주제:
1. **무게중심 균형 시뮬레이터 (`centroid_balancer`)**: 삼각형의 세 중선(꼭짓점과 대변의 중점을 잇는 선분)이 만나 언제나 일치된 교점인 무게중심($G$)을 형성하고, 분할된 6개 영역의 넓이가 삼각형 모양과 무관하게 **언제나 완벽하게 동일함**을 수학적·기하학적으로 실시간 검증합니다.
2. **프랙탈 기하학 닮음 탐색기 (`fractal_generator`)**: 부분이 전체의 형상을 무한히 복제하는 **시에르핀스키 삼각형(Sierpinski Triangle)** 프랙탈 구조를 생성하고, 마우스 휠 줌(Zoom In)을 통해 무한한 축소 시점 속에서도 완벽히 동일한 닮음의 본질이 반복되는 기하 구조를 탐색합니다.

**⚠️ 안내:** 본 파일은 파이썬 초보자도 코드 한 줄 한 줄의 기하학적·수학적 연산 원리를 이해할 수 있도록 **세포 단위의 멀티셀**로 쪼개어 구성했으며 상세한 한글 주석을 부착했습니다.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 기하학적 격자 좌표 계산을 위해 `numpy`, 인터랙티브 2D 시각화 구동을 위해 `plotly`, 그리고 제어반 조립을 위해 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 라이브러리를 먼저 준비합니다.


In [1]:
# !pip install 명령어는 주피터 노트북 내에서 터미널의 패키지 매니저를 호출하여 패키지를 설치하게 하는 매직 명령어입니다.
# -q (quiet) 옵션을 붙여 설치 로그 메시지를 간결하게 처리합니다.
%pip install -q numpy plotly ipywidgets


Note: you may need to restart the kernel to use updated packages.


### 1.2 모듈 로드 (Import)
기하 연산과 2차원 시각화 및 위젯 컴포넌트 구동에 필요한 모듈들을 메모리에 로드합니다.


In [2]:
# numpy: 다차원 좌표 배열의 연산, 중점 구하기, 면적 산출을 위한 수학 및 행렬 연산 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 마우스 인터랙션과 고화질 줌인(Zoom In) 시 좌표 깨짐 없는 벡터 플로팅을 보장하는 그래프 빌더 모듈입니다.
import plotly.graph_objects as go

# ipywidgets: 꼭짓점 좌표 슬라이더 및 프랙탈 반복 단계 슬라이더 제어반 구성을 돕는 라이브러리입니다.
import ipywidgets as widgets

# IPython.display: 면적 보고서 대시보드를 주피터 노트북 화면 내에 예쁘게 그리기 위해 로드합니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 무게중심과 면적의 평등한 조화 (`centroid_balancer`)
삼각형의 세 꼭짓점과 각각 마주 보는 변의 **중점**을 연결한 선분을 **중선(Median)**이라고 합니다. 삼각형의 세 중선은 반드시 한 점에서 만나며, 이 교점을 **무게중심(Centroid, G)**이라고 합니다.

### 무게중심의 두 가지 놀라운 수학적 성질:
1. **2:1의 내분 비례**: 무게중심 $G$는 각 중선의 길이를 꼭짓점으로부터 **$2:1$의 비율로 완벽하게 내분**합니다.
2. **면적의 6등분 평화**: 세 중선에 의해 삼각형은 6개의 작은 삼각형 영역으로 분할됩니다. 삼각형의 모양이 예각, 둔각, 직각이든 상관없이 **분할된 6개 영역의 넓이는 언제나 서로 완벽하게 똑같습니다** (전체 넓이의 $\frac{1}{6}$).

꼭짓점 A, B, C의 좌표로부터 세 중점, 무게중심 $G$, 그리고 6개 분할 삼각형의 개별 면적을 사선 공식(신발끈 공식)으로 실시간 계산하는 수학 연산 함수를 정의합니다.


In [3]:
def calculate_centroid_and_areas(A, B, C):
    """
    꼭짓점 A, B, C의 [x, y] 좌표를 입력받아 
    세 중점 D, E, F와 무게중심 G, 그리고 6개 분할 삼각형의 면적 목록을 반환합니다.
    """
    x1, y1 = A
    x2, y2 = B
    x3, y3 = C
    
    # 1. 세 변의 중점(Midpoints) 계산
    D = np.array([(x2 + x3)/2.0, (y2 + y3)/2.0]) # BC의 중점
    E = np.array([(x1 + x3)/2.0, (y1 + y3)/2.0]) # CA의 중점
    F = np.array([(x1 + x2)/2.0, (y1 + y2)/2.0]) # AB의 중점
    
    # 2. 무게중심 G(x_g, y_g) 계산: 세 꼭짓점 좌표의 산술 평균 좌표값입니다.
    G = np.array([(x1 + x2 + x3)/3.0, (y1 + y2 + y3)/3.0])
    
    # 3. 면적 연산을 위한 사선(신발끈) 공식 헬퍼 함수 정의
    # 2D 상의 세 점 P, Q, R의 면적 = 0.5 * |xp(yq - yr) + xq(yr - yp) + xr(yp - yq)|
    def get_area_3pts(P, Q, R):
        val = P[0]*(Q[1] - R[1]) + Q[0]*(R[1] - P[1]) + R[0]*(P[1] - Q[1])
        return 0.5 * abs(val)
        
    # 4. 무게중심과 각 정점/중점을 연결해 나뉜 6개 조각 삼각형의 개별 넓이 구하기
    # 6개 삼각형: GAF, GBF, GBD, GCD, GCE, GAE
    area1 = get_area_3pts(G, A, F) # 삼각형 GAF
    area2 = get_area_3pts(G, B, F) # 삼각형 GBF
    area3 = get_area_3pts(G, B, D) # 삼각형 GBD
    area4 = get_area_3pts(G, C, D) # 삼각형 GCD
    area5 = get_area_3pts(G, C, E) # 삼각형 GCE
    area6 = get_area_3pts(G, A, E) # 삼각형 GAE
    
    areas = [area1, area2, area3, area4, area5, area6]
    
    return D, E, F, G, areas


### 2.2 실시간 무게중심 분할 면적 대시보드 설계
삼각형 꼭짓점을 조절할 때마다 전체 면적 대비 분할된 6개 영역의 개별 면적 수치를 반올림하여 보여주고,
오차가 소수점 아래 무한히 수렴하여 면적이 완벽히 평등함을 증명해 주는 HTML 테이블 대시보드를 정의합니다.


In [4]:
def build_centroid_stats_html(areas, G):
    """
    6개 영역의 면적 데이터와 무게중심 G의 좌표를 시각 대시보드로 구성합니다.
    """
    total_area = sum(areas)
    average_piece_area = total_area / 6.0
    
    # 6개 면적이 얼마나 균등하게 1/6을 유지하는지 편차 검증용 텍스트 조립
    html_template = f"""
    <div style='background: rgba(255, 255, 255, 0.85);
                backdrop-filter: blur(8px);
                border: 1.5px solid rgba(11, 121, 208, 0.25);
                border-radius: 14px; padding: 20px; max-width: 600px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <h4 style='margin: 0 0 10px 0; color: #0B79D0;'>⚖️ 무게중심 G 평형 기하 분석 리포트</h4>
        <div style='margin-bottom: 12px; font-weight: bold; font-size: 1.05em;'>
            삼각형의 총 면적: <span style='color: #0B79D0;'>{total_area:.4f}</span> | 무게중심 위치: G({G[0]:.2f}, {G[1]:.2f})
        </div>
        <table style='width: 100%; border-collapse: collapse; font-size: 0.85em; text-align: center;'>
            <thead>
                <tr style='background: rgba(11, 121, 208, 0.1); border-bottom: 2px solid rgba(11, 121, 208, 0.2);'>
                    <th style='padding: 6px 4px;'>조각 1 (GAF)</th>
                    <th style='padding: 6px 4px;'>조각 2 (GBF)</th>
                    <th style='padding: 6px 4px;'>조각 3 (GBD)</th>
                    <th style='padding: 6px 4px;'>조각 4 (GCD)</th>
                    <th style='padding: 6px 4px;'>조각 5 (GCE)</th>
                    <th style='padding: 6px 4px;'>조각 6 (GAE)</th>
                </tr>
            </thead>
            <tbody>
                <tr>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[0]:.4f}</td>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[1]:.4f}</td>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[2]:.4f}</td>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[3]:.4f}</td>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[4]:.4f}</td>
                    <td style='padding: 8px 4px; font-weight: bold; color: #2FA85D;'>{areas[5]:.4f}</td>
                </tr>
                <tr style='background: rgba(47, 168, 93, 0.05); font-weight: bold; font-size: 1.05em;'>
                    <td colspan='6' style='padding: 10px; color: #2FA85D;'>
                        💡 이론상 균등 몫 (전체/6): {average_piece_area:.4f}
                    </td>
                </tr>
            </tbody>
        </table>
        <div style='margin-top: 10px; font-size: 0.85em; color: #666; font-style: italic;'>
            📌 <b>관찰 포인트:</b> 꼭짓점의 위치를 아무리 일그러뜨려 비대칭적인 삼각형으로 찌그러뜨려도, 6개 분할 영역의 개별 넓이는 소수점 아래 넷째 자리까지 오차 없이 <b>정확히 동일한 값</b>을 사수합니다. 중선들의 2:1 조화와 넓이의 평화로운 동등성을 기하학적으로 입증합니다.
        </div>
    </div>
    """
    return html_template


### 2.3 Plotly 2D 동적 무게중심 및 6분할 컬러 매핑
6개의 쪼개진 삼각형 영역을 각각 다른 반투명 파스텔톤 다각형(go.Scatter fill='toself')으로 덮어 그립니다.
또한 중선 뼈대와 무게중심(G), 꼭짓점 A, B, C를 오버레이하여 역학적 시뮬레이터 플롯을 구축합니다.


In [5]:
def draw_centroid_balancer_scene(A_x, A_y, B_x, B_y, C_x, C_y):
    """
    꼭짓점 좌표 입력값에 맞추어 무게중심 작도 및 6색 영역 분할을 플로팅합니다.
    """
    A = np.array([A_x, A_y])
    B = np.array([B_x, B_y])
    C = np.array([C_x, C_y])
    
    # 1. 중점, 무게중심, 면적 데이터 산출
    D, E, F, G, areas = calculate_centroid_and_areas(A, B, C)
    
    # 2. HTML 통계 대시보드 리포팅 표출
    display(HTML(build_centroid_stats_html(areas, G)))
    
    traces = []
    
    # 3. 6개 조각 삼각형의 반투명 채우기 영역 정의 (go.Scatter의 fill='toself' 기능 적용)
    # 조각들의 꼭짓점 연결 리스트 정보
    slices_vertices = [
        ([G[0], A[0], F[0], G[0]], [G[1], A[1], F[1], G[1]], 'GAF', 'rgba(11, 121, 208, 0.25)'),
        ([G[0], B[0], F[0], G[0]], [G[1], B[1], F[1], G[1]], 'GBF', 'rgba(47, 168, 93, 0.25)'),
        ([G[0], B[0], D[0], G[0]], [G[1], B[1], D[1], G[1]], 'GBD', 'rgba(245, 158, 11, 0.25)'),
        ([G[0], C[0], D[0], G[0]], [G[1], C[1], D[1], G[1]], 'GCD', 'rgba(225, 29, 72, 0.25)'),
        ([G[0], C[0], E[0], G[0]], [G[1], C[1], E[1], G[1]], 'GCE', 'rgba(139, 92, 246, 0.25)'),
        ([G[0], A[0], E[0], G[0]], [G[1], A[1], E[1], G[1]], 'GAE', 'rgba(20, 184, 166, 0.25)')
    ]
    
    for x_pts, y_pts, name, color in slices_vertices:
        traces.append(go.Scatter(
            x=x_pts, y=y_pts,
            fill='toself',      # 닫힌 영역 내부 채우기 활성화
            fillcolor=color,    # 영역별 은은한 파스텔 투명 색 지정
            mode='lines',
            line=dict(width=0.5, color='rgba(0,0,0,0.1)'), # 얇은 테두리
            name=name,
            hoverinfo='none'
        ))
        
    # 4. 세 중선(Median Lines) 골격 그리기
    # 꼭짓점과 대변 중점 잇기: A->D, B->E, C->F (선 끊김을 위해 None 사용)
    traces.append(go.Scatter(
        x=[A[0], D[0], None, B[0], E[0], None, C[0], F[0]],
        y=[A[1], D[1], None, B[1], E[1], None, C[1], F[1]],
        mode='lines',
        line=dict(color='#475569', width=2.5, dash='dash'), # 어두운 회색 점선
        name='중선 (Median Lines)',
        hoverinfo='skip'
    ))
    
    # 5. 외각 삼각형 ABC 굵은 테두리 뼈대 그리기
    traces.append(go.Scatter(
        x=[A[0], B[0], C[0], A[0]], y=[A[1], B[1], C[1], A[1]],
        mode='lines+markers',
        line=dict(color='#1E2937', width=4.5), # 굵고 선명한 테두리
        marker=dict(size=12, color='#1E2937'),
        name='삼각형 ABC'
    ))
    
    # 6. 무게중심 G 마커 (별 모양) 오버레이
    traces.append(go.Scatter(
        x=[G[0]], y=[G[1]],
        mode='markers+text',
        marker=dict(size=16, color='#E11D48', symbol='star-diamond', line=dict(color='white', width=1)),
        text=['무게중심 (G)'], textposition='top center',
        name='무게중심 (G)',
        hoverinfo='x+y'
    ))
    
    # 7. 레이아웃 설계 (정비율 1:1 격자 설정)
    layout = go.Layout(
        title=dict(text='<b>삼각형 무게중심 및 6분할 균등 면적 시뮬레이션</b>', x=0.5),
        xaxis=dict(range=[-10, 10], gridcolor='#F1F5F9', zeroline=True, zerolinecolor='#CBD5E1'),
        yaxis=dict(range=[-10, 10], gridcolor='#F1F5F9', scaleanchor='x', scaleratio=1.0, zeroline=True, zerolinecolor='#CBD5E1'),
        plot_bgcolor='white',
        width=650, height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98, bordercolor='#E2E8F0', borderwidth=1)
    )
    
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


### 2.4 무게중심 역학 시뮬레이터 구동
삼각형의 세 정점의 x, y 좌표들을 독립적으로 이동시킬 수 있는 6개의 제어 슬라이더 패널을 열어 시뮬레이터를 조작합니다.
각 슬라이더를 드래그해 삼각형의 모양을 심하게 찌그러뜨렸을 때도 6개 다각형의 면적 테이블이 평등을 유지하고,
각 중선 뼈대 대칭선 한가운데 무게중심이 굳건히 자리 잡는 물리적 질서를 탐색합니다.


In [6]:
# 꼭짓점 제어 슬라이더 생성 (x, y 범위 -7에서 +7)
slider_Ax = widgets.FloatSlider(value=0.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 A_x:')
slider_Ay = widgets.FloatSlider(value=6.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 A_y:')

slider_Bx = widgets.FloatSlider(value=-6.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 B_x:')
slider_By = widgets.FloatSlider(value=-4.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 B_y:')

slider_Cx = widgets.FloatSlider(value=6.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 C_x:')
slider_Cy = widgets.FloatSlider(value=-3.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 C_y:')

# 정점별로 VBox 조립
controls_box = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0;'> 꼭짓점 A 조작</h4>"), slider_Ax, slider_Ay,
    widgets.HTML("<h4 style='color:#2FA85D; margin:0;'> 꼭짓점 B 조작</h4>"), slider_Bx, slider_By,
    widgets.HTML("<h4 style='color:#E11D48; margin:0;'> 꼭짓점 C 조작</h4>"), slider_Cx, slider_Cy
])

# 위젯 신호와 렌더링 엔진 바인딩
output_scene = widgets.interactive_output(
    draw_centroid_balancer_scene,
    {
        'A_x': slider_Ax, 'A_y': slider_Ay,
        'B_x': slider_Bx, 'B_y': slider_By,
        'C_x': slider_Cx, 'C_y': slider_Cy
    }
)

# 가로 방향으로 대시보드 표출
display(widgets.HBox([controls_box, output_scene]))


---
## 3. 실습 2: 프랙탈 기하학과 닮음의 무한성 (`fractal_generator`)
기하학적 **닮음(Similarity)**의 대상을 극단으로 확장하면, 아주 작은 일부분의 모양이 전체의 모양과 끊임없이 닮아 있는 **자기닮음성(Self-Similarity)**을 품은 **프랙탈(Fractal)**의 신비로운 구조와 마주하게 됩니다.

### 시에르핀스키 삼각형 (Sierpinski Triangle) 생성 원리:
1. **0단계**: 큰 정삼각형 하나를 그립니다.
2. **1단계**: 삼각형의 세 변의 중점을 이어 생기는 가운데 뒤집힌 삼각형 부분을 파냅니다 (1개 제거, 3개의 닮음 삼각형 생성).
3. **N단계**: 남은 주변의 3개 삼각형 각각에 대해 중점을 이어 파내는 과정을 무한히 재귀(Recursion) 반복합니다.

이 단계별 프랙탈 다각형 꼭짓점 체인 리스트를 재귀적 알고리즘으로 계산하는 파이썬 생성 함수를 정의합니다.


In [7]:
def generate_sierpinski_triangles(depth):
    """
    지정된 재귀 단계(depth)에 대응하는 시에르핀스키 삼각형의 
    개별 서브 삼각형들의 꼭짓점 리스트들을 계산하여 리턴합니다.
    """
    # 1. 0단계 기준 정삼각형의 세 꼭짓점 정의
    # 한 변의 길이가 10이고 바닥이 (-5, 0), (5, 0)인 정삼각형 꼭짓점
    A = np.array([0.0, 5 * np.sqrt(3)])
    B = np.array([-5.0, 0.0])
    C = np.array([5.0, 0.0])
    
    triangle_list = []
    
    # 2. 재귀 분할 함수 정의
    def subdivide(p1, p2, p3, current_depth):
        # 목표 단계에 다다르면 현재 세 점으로 구성된 삼각형을 결과 리스트에 추가합니다.
        if current_depth == 0:
            triangle_list.append(np.array([p1, p2, p3, p1])) # 닫힌 루프를 위해 p1을 끝에 추가
            return
            
        # 각 변의 중점을 구합니다.
        m12 = (p1 + p2) / 2.0
        m23 = (p2 + p3) / 2.0
        m31 = (p3 + p1) / 2.0
        
        # 가운데 삼각형(m12, m23, m31)을 파내고 남은 주변 3개의 하위 삼각형에 대해 
        # 재귀적으로 다시 분할을 지시합니다 (depth - 1)
        subdivide(p1, m12, m31, current_depth - 1)  # 상단 삼각형
        subdivide(m12, p2, m23, current_depth - 1)  # 좌하단 삼각형
        subdivide(m31, m23, p3, current_depth - 1)  # 우하단 삼각형
        
    # 재귀 함수를 호출하여 리스트를 채웁니다.
    subdivide(A, B, C, depth)
    return triangle_list


### 3.2 벡터 줌인을 보장하는 Plotly 프랙탈 렌더러 설계
Plotly 2D 다각 렌더링에 적합하도록 각 서브 삼각형들을 하나의 트레이스(`go.Scatter` fill='toself')로 묶거나,
또는 전체를 하나의 벡터 트레이스로 엮어 마우스 휠로 확대(Zoom In)해도 화질 깨짐 없이 무한한 축소 삼각형들이 드러나도록 렌더러 엔진을 설계합니다.
면의 색상에는 브랜드 블루 컬러(`#0B79D0`)와 그린 컬러(`#2FA85D`)를 조합한 그라데이션 컨셉을 반영합니다.


In [8]:
def render_sierpinski_simulation(depth):
    """
    재귀 단계(depth)에 맞춰 연산된 시에르핀스키 삼각형의 좌표들을 
    Plotly 평면 상에 동적으로 렌더링합니다.
    """
    # 1. 재귀 기하 정점들 획득
    triangles = generate_sierpinski_triangles(depth)
    
    # 2. 단계 정보 테이블 대시보드 출력
    sub_triangle_count = 3 ** depth
    scale_factor = (0.5) ** depth # 닮음비 (각 단계별 1/2배씩 축소)
    area_ratio = (0.25) ** depth  # 면적비 (닮음비의 제곱, 1/4배씩 축소)
    
    info_html = f"""
    <div style='background: white; border: 1.5px solid rgba(11, 121, 208, 0.2);
                padding: 16px; border-radius: 12px; max-width: 600px; margin-bottom: 15px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <h4 style='margin: 0 0 8px 0; color: #0B79D0;'>🌀 시에르핀스키 프랙탈 단계 분석 ({depth}단계)</h4>
        <div style='display: flex; justify-content: space-around; text-align: center; font-size: 0.9em;'>
            <div>
                <span style='color: #666;'>나뉜 삼각형 수</span>
                <h3 style='margin: 5px 0 0 0; color: #0B79D0;'>{sub_triangle_count} 개</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #666;'>하위 닮음비 (Scale)</span>
                <h3 style='margin: 5px 0 0 0; color: #475569;'>{scale_factor:.4f} 배</h3>
            </div>
            <div style='border-left: 1px solid #E2E8F0; padding-left: 20px;'>
                <span style='color: #666;'>개별 면적비 (Area)</span>
                <h3 style='margin: 5px 0 0 0; color: #2FA85D;'>{area_ratio:.6f} 배</h3>
            </div>
        </div>
    </div>
    """
    display(HTML(info_html))
    
    # 3. Plotly 2D 다각형 그리기 구성
    traces = []
    
    # 성능 낭비를 막고 고해상도 벡터 렌더링을 구현하기 위해,
    # 모든 서브 삼각형 꼭짓점들의 x, y 좌표 사이에 None을 끼워 넣어 하나의 Scatter Trace로 그립니다.
    all_x, all_y = [], []
    for tri in triangles:
        all_x.extend(list(tri[:, 0]) + [None])
        all_y.extend(list(tri[:, 1]) + [None])
        
    # go.Scatter의 fill='toself' 옵션과 묶음 궤적으로 속도가 매우 빠르고 
    # 휠 줌을 극단적으로 당겨도 찌그러짐 없는 무한 벡터 줌인 기하가 완성됩니다.
    traces.append(go.Scatter(
        x=all_x, y=all_y,
        fill='toself',
        fillcolor='rgba(11, 121, 208, 0.75)', # 브랜드 블루 투명도 75%
        mode='lines',
        line=dict(color='#2FA85D', width=1.5),  # 테두리는 브랜드 그린선
        name=f'{depth}단계 삼각형들',
        hoverinfo='skip'
    ))
    
    # 레이아웃 정의
    layout = go.Layout(
        title=dict(
            text='<b>Sierpinski Triangle 무한 닮음 프랙탈 탐색기</b><br><span style="font-size:11px;color:#666;">마우스 휠 스크롤: 무한 확대(Zoom In)로 내부의 무한 닮음 반복을 통찰해 보세요</span>',
            x=0.5
        ),
        xaxis=dict(range=[-6, 6], gridcolor='#F1F5F9', zeroline=False),
        yaxis=dict(range=[-1, 9], gridcolor='#F1F5F9', scaleanchor='x', scaleratio=1.0, zeroline=False),
        plot_bgcolor='white',
        width=700, height=600,
        margin=dict(l=30, r=30, b=40, t=65),
        showlegend=False
    )
    
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


### 3.3 프랙탈 제어 슬라이더 구동
재귀 반복 단계(Depth)를 0(기본 정삼각형)부터 5단계까지 가변 조작하는 정수 슬라이더 위젯을 조립하여 실행합니다.
단계를 올릴수록 닮음 조각들이 폭발적으로 분열(나뉜 삼각형 개수: $3^{\text{depth}}$)하는 스케일업 현상을 조작하고,
마우스 휠을 당겨 거대함 속에 극소함이 동일한 AA 닮음 각도로 무한 루프하는 프랙탈의 내면을 확대 관찰합니다.


In [9]:
# 프랙탈 재귀 단계를 가변 조작하는 위젯 정의 (연산 부하를 고려해 0단계에서 최대 5단계로 제어 제한)
fractal_depth_slider = widgets.IntSlider(
    value=2, # 초기값: 2단계
    min=0, max=5, step=1,
    description='프랙탈 반복 단계(Depth):',
    style={'description_width': 'initial'},
    continuous_update=False # 마우스 클릭을 떼었을 때 재귀 계산을 돌려 화면 갱신 최적화
)

# 위젯 바인딩 기동
widgets.interact(render_sierpinski_simulation, depth=fractal_depth_slider);


interactive(children=(IntSlider(value=2, continuous_update=False, description='프랙탈 반복 단계(Depth):', max=5, styl…

---
## 4. 깊이 있는 기하학 사색을 위한 열린 질문 (Joshua를 위한 질문)

1. **조직 분할과 무게중심의 평등 법칙**:
   삼각형을 불규칙적으로 일그러뜨렸을 때도, 중선들의 기하학적 당김에 의해 6개의 분할 공간 면적은 항상 완벽히 균등한 $\frac{1}{6}$을 사수했습니다.
   조직의 형태나 이해관계(꼭짓점의 위치)가 격렬하게 요동칠 때도 조직의 부서 간 균형과 에너지 분배가 완벽한 평등을 유지하게 돕는 리더십적 '중선(꼭짓점과 대변 중점을 잇는 다리)'과 '무게중심적 원칙'은 무엇에 비유될 수 있을까요? 역학적인 조화와 평화에 대해 사색해 보고 느낀 소회를 남겨 주세요.

2. **무한 줌인(Zoom In) 속 프랙탈의 일관성**:
   시에르핀스키 삼각형의 심연 속으로 마우스 휠 줌을 끝없이 당겨도, 그 내부에는 0단계의 거대한 시작 삼각형과 한 치의 비례 오차도 없는 닮은꼴 삼각형들의 무한 우주가 끊임없이 펼쳐졌습니다.
   조직이나 비즈니스의 스케일이 극도로 작아지거나(스타트업/마이크로 태스크), 반대로 무한히 커질(글로벌 대기업/거대 비전) 때도 결코 훼손되지 않고 무한히 반복하여 재현되어야 할 내 비즈니스의 '본질적인 핵심 DNA(프랙탈 단위 구조)'는 무엇인가요? 기하학의 무한 루프를 보며 사색해 보세요.
